In [1]:
import pandas as pd
import numpy as np
import re
import csv
import os

## generate 3-to-1 letter AA codes

In [ ]:
# Mapping from three-letter to one-letter amino acid codes
three_to_one = {
    'Ala': 'A', 'Cys': 'C', 'Asp': 'D', 'Glu': 'E', 'Phe': 'F', 'Gly': 'G', 'His': 'H',
    'Ile': 'I', 'Lys': 'K', 'Leu': 'L', 'Met': 'M', 'Asn': 'N', 'Pro': 'P', 'Gln': 'Q',
    'Arg': 'R', 'Ser': 'S', 'Thr': 'T', 'Val': 'V', 'Trp': 'W', 'Tyr': 'Y'
}

def convert_three_to_one(mutation):
    # Modify the regex to match the gene name but ignore it in the output
    match = re.match(r"[a-zA-Z0-9]+_p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})", mutation)
    if not match:
        return None
    original_aa, position, new_aa = match.groups()
    try:
        # Return just the one-letter mutation string
        return f"p.{three_to_one[original_aa]}{position}{three_to_one[new_aa]}"
    except KeyError:
        return None

# Example usage:
print(convert_three_to_one("atpE_p.Asp28Gly"))  # Output: p.D28G
print(convert_three_to_one("atpE_p.Ala249Thr"))  # Output: p.A249T


In [ ]:
# check if the genes of interest are in catalog df
protein_sequences_file = '/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/catalog/protein_sequences.csv'
protein_sequences_df = pd.read_csv(protein_sequences_file)
# Extract genes of interest
genes_of_interest = protein_sequences_df['gene'].unique()

In [ ]:
## load the 2023 complete excel file

# Load the WHO catalog
who_catalog_path = '/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/caatalog/HO-UCN-TB-2023.7-eng.xlsx'  # Update the path as necessary
who_catalog = pd.read_excel(who_catalog_path, sheet_name='Catalogue_master_file',header=2)

In [ ]:
# Extract relevant columns
catalog_df = who_catalog[['drug','gene', 'mutation','variant','effect','Present_R','Present_S','FINAL CONFIDENCE GRADING']]
print(catalog_df.head())

# Filter out rows with NaN values in the 'drug' or 'variant' columns
catalog_df = catalog_df.dropna(subset=['drug', 'variant'])
print("len of frequency row", len(frequency_df))

# subset for target genes
filtered_df = catalog_df[catalog_df['variant'].str.startswith(tuple(genes_of_interest))]

# Display the filtered DataFrame
print(len(filtered_df))
# Discard rows that have 'ins' or 'del' in the variant column
filtered_df = filtered_df[~filtered_df['variant'].str.contains('ins|del')]

In [ ]:
filtered_df=filtered_df.rename(columns={"FINAL CONFIDENCE GRADING": "confidence"})

In [ ]:
## apply the 3-to-1 conversion
filtered_df['one_letter_mutation'] = filtered_df['variant'].apply(convert_three_to_one)

In [ ]:
filtered_df

In [ ]:
## drop invalid mutations
filtered_df = filtered_df.dropna(subset=['one_letter_mutation'])

In [ ]:
print(np.unique(filtered_df['gene']))

In [ ]:
filtered_df =filtered_df.drop_duplicates()

In [ ]:
filtered_df.to_csv('/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/catalog/mutations_with_one_letter_all_confidence.csv', index=False)

## feature 1: frequency

In [ ]:
# Calculate the frequency of each variant in the population
filtered_df['frequency'] = (filtered_df['Present_R']) / (filtered_df['Present_R'] + filtered_df['Present_S'])

In [ ]:
filtered_df

In [ ]:
filtered_df.drop(columns=['Present_R','Present_S'], inplace=True)

In [ ]:
filtered_df

In [ ]:
filtered_df.to_csv('/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/all_proteins_frequency_catalog.csv', index=False)

## generate mutated sequences from the WHO catalog

In [ ]:
protein_sequences_df = pd.read_csv('/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/catalog/protein_sequences.csv')
# Output directory
output_dir = '/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/mutated_sequences'
# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

In [ ]:
catalog_df=pd.read_csv('/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/all_proteins_frequency_catalog.csv')

In [ ]:
# Function to apply a mutation
# produce mutated seq
def apply_mutation(wildtype_seq, mutation):
    mutation_info = mutation.split('.')
    if len(mutation_info) != 2:
        return None
    mutation_type, change = mutation_info
    if mutation_type != 'p':
        return None
    original_aa = change[0]
    position = int(''.join(filter(str.isdigit, change)))
    new_aa = change[-1]
    if position <= 0 or position > len(wildtype_seq):
        return None
    mutated_seq = wildtype_seq[:position - 1] + new_aa + wildtype_seq[position:]
    return original_aa, new_aa, position, mutated_seq

In [ ]:
# Process each protein sequence and produce mutated seq
for index, row in protein_sequences_df.iterrows():
    gene = row['gene']
    rv_id = row['RV']
    wildtype_seq = row['protein_sequence']
    
    # Output FASTA file for the gene
    output_file = os.path.join(output_dir, f"{rv_id}_{gene}.fasta")
    
    with open(output_file, mode='w') as file:
        # Write wildtype sequence
        file.write(f">{rv_id}|{gene}|Wildtype\n")
        file.write(f"{wildtype_seq}\n")
        
        # Get mutations for the specific gene
        gene_mutations = catalog_df[catalog_df['gene'] == gene]
        
        for _, mut_row in gene_mutations.iterrows():
            mutation = mut_row['one_letter_mutation']
            result = apply_mutation(wildtype_seq, mutation)
            if result:
                original_aa, new_aa, position, mutated_seq = result
                mutation_label = f"{gene}_p.{original_aa}{position}{new_aa}"
                file.write(f">{rv_id}|{gene}|{mutation_label}\n")
                file.write(f"{mutated_seq}\n")
                print(f"Mutation {mutation} applied for gene {gene}.")

## feature 2: compute delta-z value from the mutated seqs

In [ ]:
# call delta_z_calculation file

In [ ]:
import torch
import numpy as np
import pandas as pd
import esm
from Bio import SeqIO

In [ ]:
# Function to calculate embeddings
def get_embedding(sequence):
    sequence = sequence.upper()  # Ensure sequence is in uppercase
    sequence = sequence.replace('*', '')  # Remove any stop codons
    batch_labels, batch_strs, batch_tokens = batch_converter([("sequence", sequence)])
    with torch.no_grad():
        results = model(batch_tokens.to('cuda'), repr_layers=[6])
    token_embeddings = results["representations"][6].cpu().numpy()
    return token_embeddings.mean(axis=1)

In [ ]:
# Load the pretrained ESM-2 model
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval().to('cuda')

In [ ]:
# Directory containing the FASTA files
fasta_dir = '/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/mutated_sequences'

# Output CSV file
output_file = '/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/all_delta_z_values.csv'

In [ ]:
# Initialize the results list
results = []

# Process each FASTA file
for fasta_file in os.listdir(fasta_dir):
    if fasta_file.endswith('.fasta'):
        file_path = os.path.join(fasta_dir, fasta_file)
        records = list(SeqIO.parse(file_path, "fasta"))
        print("loaded file: ", file_path)
        
        # Extract RV and gene names from the file name
        rv_id, gene_name = fasta_file.replace('.fasta', '').split('_')
        print("continue")
        
        # Find the wildtype sequence
        wildtype_record = None
        for record in records:
            if "Wildtype" in record.description:
                wildtype_record = record
                break
        
        if wildtype_record is None:
            print(f"No wildtype sequence found in {fasta_file}")
            continue
        
        wildtype_seq = str(wildtype_record.seq).upper()
        wildtype_embedding = get_embedding(wildtype_seq)
        
        for record in records:
            if record.description == wildtype_record.description:
                continue
            mutation = record.description
            mutated_seq = str(record.seq).upper().replace('*', '')
            try:
                mutated_embedding = get_embedding(mutated_seq)
                delta_z = np.linalg.norm(mutated_embedding - wildtype_embedding)
                results.append([fasta_file, mutation, delta_z, rv_id, gene_name])
            except KeyError as e:
                print(f"Error processing {mutation} in {fasta_file}: {e}")

In [ ]:
# Save the results to a CSV file
results_df = pd.DataFrame(results, columns=['filename', 'mutation', 'delta_z', 'rv', 'gene'])
results_df.to_csv(output_file, index=False)

print(f"Delta Z values have been saved to {output_file}.")


In [ ]:
## delta-z calculates the euclidean distance between wildtype seq and mutated sequence embeddings
## embeddings are produced from pretrained ESM2(esm2_t6_8M_UR50D)

### add delta-z to the catalog_df

In [ ]:
catalog_df=pd.read_csv('/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/all_proteins_frequency_catalog.csv')
delta_z_df=pd.read_csv('/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/all_delta_z_values.csv')

In [ ]:
catalog_df=catalog_df.drop_duplicates()
delta_z_df=delta_z_df.drop_duplicates()

In [ ]:
# Remove the gene name from the beginning of the 'Mutation' column
delta_z_df['mutation'] = delta_z_df['mutation'].apply(lambda x: x.split('_')[1] if '_' in x else x)
# Ensure the 'Mutation' column is a string and handle missing values
delta_z_df['mutation'] = delta_z_df['mutation'].astype(str).fillna('')

In [ ]:
# Merge the filtered Delta-Z DataFrame with the filtered mutations DataFrame to get the additional columns
merged_df = pd.merge(delta_z_df, catalog_df, left_on=['gene', 'mutation'], right_on=['gene', 'one_letter_mutation'], how='left')

In [ ]:
merged_df

In [ ]:
merged_df=merged_df.drop(columns=['mutation_x','mutation_y','variant','rv','filename'])

In [ ]:
merged_df=merged_df.drop_duplicates()

In [ ]:
merged_df['one_letter_mutation'] = merged_df['one_letter_mutation'].str.replace('p.', '')

In [ ]:
# Define the desired order of columns, placing 'gene' first
cols = ['gene', 'one_letter_mutation', 'drug', 'confidence','delta_z', 'frequency']

# Reorder the DataFrame columns
reordered_df = merged_df[cols]

# Display or use the reordered DataFrame as needed
reordered_df

In [ ]:
reordered_df.to_csv('/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/all_proteins_freq_deltaz.csv',index=False)

## feature 3: proximity to nearest r-conferring mutations

### helper codes

In [ ]:
def extract_position_from_mutation(mutation):
    match = re.search(r'\d+', mutation)
    if match:
        return int(match.group(0))
    return None
def adjust_number(index):
    if index < 30:  # Assuming offset starts at position 23 + 7
        return index + 7
    elif 30 <= index <= 1180:  # Adjusts up to the maximum offset point
        return index + 6
    else:
        return index
def non_self_proximity_r_mutants(gene_subset,unique_valid_r_positions,distance_map):
    # Initialize a list to collect positions causing KeyError
    positions_causing_error = []
    # Initialize the list to store the minimum distances and their indices
    proximity_to_nonself_r_conferring = []
    nearest_mutation_index = []

    for index, row in gene_subset.iterrows():
        current_position = row['position']
        adjusted_current_position = adjust_number(current_position)
        distances = []

        for r_pos in unique_valid_r_positions:
            if r_pos != current_position:
                adjusted_r_pos = adjust_number(r_pos)
                try:
                    dist = distance_map.dist(adjusted_current_position, adjusted_r_pos, raise_na=True)
                    if not np.isnan(dist):
                        distances.append((dist, r_pos))
                except KeyError as e:
                    # Append the positions causing the KeyError
                    positions_causing_error.append((current_position, r_pos))
                    continue  # Continue with the next position

        if distances:
            # Find the entry with the minimum distance
            min_distance, min_index = min(distances, key=lambda x: x[0])
        else:
            min_distance, min_index = np.nan, np.nan

        # Append results to the lists
        proximity_to_nonself_r_conferring.append(min_distance)
        nearest_mutation_index.append(min_index)
    # Store the results in the phenotype_data DataFrame
    gene_subset['Proximity_to_R_Conferring'] = proximity_to_nonself_r_conferring
    gene_subset['Nearest_Mutation_Index'] = nearest_mutation_index
    
    return gene_subset


### compute proximity

In [22]:
from evcouplings.compare import DistanceMap

In [ ]:
catalog_df=pd.read_csv('/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/all_proteins_freq_deltaz.csv')

In [ ]:
# Map confidence levels to phenotypes
def map_confidence_to_phenotype(confidence):
    if confidence in ['1) Assoc w R', '2) Assoc w R - Interim']:
        return 'Resistant'
    elif confidence in ['4) Not assoc w R - Interim', '5) Not assoc w R']:
        return 'Susceptible'
    else:
        return 'Unknown'


In [ ]:
catalog_df['phenotype'] =catalog_df['confidence'].apply(map_confidence_to_phenotype)

In [ ]:
catalog_df['phenotype'].value_counts()

In [ ]:
catalog_df['position'] =catalog_df['one_letter_mutation'].str.extract(r'(\d+)').astype(int)

In [ ]:
# Get the unique combinations of gene and drug
unique_genes_drugs = catalog_df[['gene', 'drug']].drop_duplicates()

# Print each gene and drug pair
for index, row in unique_genes_drugs.iterrows():
    print(f"Gene: {row['gene']}, Drug: {row['drug']}")

In [ ]:
gene_names=list(np.unique(catalog_df['gene']))

In [19]:
##load the protein details file
protein_details_path = '/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/catalog/17_proteins_details.xlsx'  # Update the path as necessary
protein_details = pd.read_excel(protein_details_path, sheet_name='Sheet1')

In [ ]:
protein_details

In [ ]:
# Filter the DataFrame for the genes of interest
filtered_df = protein_details[protein_details['gene_name'].str.contains('|'.join(gene_names), case=False, na=False)]

In [ ]:
# Initialize an empty list to store DataFrames
all_gene_data = []
for index, row in filtered_df.iterrows():
    # Extract data from the current row
    fasta_filename = row['filename']
    print(f"Processing file: {fasta_filename}")
    uniprot = row['Uniprot']
    entry = row['Entry']
    drug = row['drug_full']
    drug_code = row['Drug']
    gene_name = row['gene_name']
    print("calculating for gene:", gene_name)
    
    ## calculate proximity
    gene_subset=catalog_df[catalog_df['gene'] == gene_name]
    
    ## get r-conferring mutation positions
    r_conferring_mutations = gene_subset[gene_subset['phenotype'] == 'Resistant']['one_letter_mutation'].tolist()
    r_conferring_positions = [extract_position_from_mutation(mutation) for mutation in r_conferring_mutations]
    
    ##load distance file
    distmap_path = f"/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/distmaps/{uniprot}/{entry}"
    dist_map = DistanceMap.from_file(distmap_path)
    
    # Filter valid positions
    valid_r_positions = [pos for pos in r_conferring_positions if pos < dist_map.dist_matrix.shape[0]]
    unique_valid_r_positions=np.unique(valid_r_positions)
    print("len unique Valid R-Conferring Positions:", len(unique_valid_r_positions))
    
    gene_subset=gene_subset.drop_duplicates()

    # # Debug print to check the filtered positions
    print("len Valid R-Conferring Positions:", len(valid_r_positions))
    
    gene_subset=non_self_proximity_r_mutants(gene_subset,unique_valid_r_positions,dist_map)
    # Append the processed DataFrame to the list
    all_gene_data.append(gene_subset)

In [ ]:
# After the loop, concatenate all the DataFrames in the list into a single large DataFrame
final_df = pd.concat(all_gene_data, ignore_index=True)

In [ ]:
final_df

In [ ]:
final_df.to_csv("/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/all_proteins_freq_details_proximity.csv",index=False)

## feature 4: amino acid index dist

In [ ]:
### essential methods

# Function to extract amino acid and position from mutation string
def extract_aa_and_position(mutation):
    match = re.match(r'([A-Z])(\d+)([A-Z])', mutation)
    if match:
        return match.groups()
    return None, None, None

# Function to calculate the Euclidean distance between two amino acids
def calculate_euclidean_distance(wt_aa, mutant_aa, amino_acid_indices):
    if wt_aa in amino_acid_indices and mutant_aa in amino_acid_indices:
        wt_indices = np.array(amino_acid_indices[wt_aa])
        mutant_indices = np.array(amino_acid_indices[mutant_aa])
        distance = np.linalg.norm(wt_indices - mutant_indices)
        return distance
    else:
        print(f"Missing indices for {wt_aa} or {mutant_aa}")
        return None


In [ ]:
catalog_df=pd.read_csv("/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/all_proteins_freq_details_proximity.csv")

In [ ]:
# Load the CSV file into a DataFrame
amino_acid_data_path = '/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/AAIndex_PCA.csv'  # Update with the correct path
amino_acid_df = pd.read_csv(amino_acid_data_path, index_col=0)

In [ ]:
all_gene_data = []
for index, row in catalog_df.iterrows():
    # Extract data from the current row
    gene_name = row['gene']
    print("calculating for gene:", gene_name)
    
    ## calculate aa index dits
    gene_subset=catalog_df[catalog_df['gene'] == gene_name]
    
    mutations = gene_subset['one_letter_mutation']

    # Create a dictionary to map amino acids to their corresponding index values
    amino_acid_indices = amino_acid_df.set_index(amino_acid_df.index).T.to_dict('list')
    # Calculate distances for each mutation
    distances = []
    for mutation in mutations:
        wt_aa, position, mutant_aa = extract_aa_and_position(mutation)
        if wt_aa and mutant_aa:
            distance = calculate_euclidean_distance(wt_aa, mutant_aa, amino_acid_indices)
            distances.append((distance))
    # Add the distances as a new column to the phenotype data
    gene_subset['aa_index_dist'] = distances
    all_gene_data.append(gene_subset)

In [ ]:
# After the loop, concatenate all the DataFrames in the list into a single large DataFrame
final_df = pd.concat(all_gene_data, ignore_index=True)

In [252]:
final_df.to_csv("/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/all_proteins_freq_details_proximity_aaindex.csv",index=False)

## feature 5: log likelihood ratio

In [72]:
from transformers import AutoTokenizer, EsmForMaskedLM
import torch

In [73]:
catalog_df=pd.read_csv("/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/all_proteins_freq_details_proximity_aaindex.csv")

In [ ]:
catalog_df=catalog_df.drop_duplicates()

### debugging fgd1 and pepq

In [15]:
# Filter rows where proximity is 0.0
proximity_zero_rows = catalog_df[catalog_df['Proximity_to_R_Conferring'] == 'NaN']

# Display the rows with proximity = 0.0
print("Rows where proximity is 0.0:")
print(proximity_zero_rows)

# Get the value counts of phenotype for proximity = 0.0
phenotype_counts = proximity_zero_rows['phenotype'].value_counts()

# Display the counts
print("\nPhenotype value counts where proximity is 0.0:")
print(phenotype_counts)

Rows where proximity is 0.0:
Empty DataFrame
Columns: [gene, one_letter_mutation, drug, confidence, delta_z, frequency, phenotype, position, Proximity_to_R_Conferring, Nearest_Mutation_Index, aa_index_dist]
Index: []

Phenotype value counts where proximity is 0.0:
Series([], Name: count, dtype: int64)


In [ ]:
# Filter rows where 'Proximity_to_R_Conferring' is NaN
nan_proximity_rows = catalog_df[catalog_df['Proximity_to_R_Conferring'].isna()]

# Print the filtered rows
print(nan_proximity_rows)


In [18]:
# Count the number of rows with NaN 'Proximity_to_R_Conferring' for each gene
nan_proximity_count = catalog_df[catalog_df['Proximity_to_R_Conferring'].isna()].groupby('gene').size().reset_index(name='NaN Count')

# Print the result
print(nan_proximity_count)


      gene  NaN Count
0   Rv0678         18
1     atpE          1
2      ddn          2
3     embB         33
4     ethA          6
5     fgd1        107
6      gid         21
7     gyrA         67
8     gyrB         35
9     inhA         13
10    katG         20
11    pepQ        131
12    pncA          9
13    rplC          3
14    rpoB         11
15    rpsL          2
16    tlyA          3


In [51]:
## filter by gene
## calculate by gene
gene_name = 'pepQ'
genes_of_interest =gene_name.split(',')
print(f"Genes of interest: {genes_of_interest}")
gene_subset=catalog_df[catalog_df['gene'] == gene_name]
## get r-conferring mutation positions
r_conferring_mutations = gene_subset[gene_subset['phenotype'] == 'Resistant']['one_letter_mutation'].tolist()
r_conferring_positions = [extract_position_from_mutation(mutation) for mutation in r_conferring_mutations]
print(len(np.unique(r_conferring_mutations)),len(np.unique(r_conferring_positions)))
print(np.unique(r_conferring_positions))

# # Load the 3D distance map
# if gene_name == 'fgd1':
#     distmap_path = f"/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/proteins_for_mahbuba/FGD1_MYCTU/P9WNE1"
# distance_map = DistanceMap.from_file(distmap_path)


Genes of interest: ['pepQ']
0 0
[]


In [ ]:
distance_map.residues_i

,id,seqres_id,coord_id,one_letter_code,three_letter_code,chain_index,chain_id,sec_struct,sec_struct_3state,hetatm
0,3,23,3,E,GLU,0,A,C,C,False
1,4,24,4,L,LEU,0,A,C,C,False
2,5,25,5,K,LYS,0,A,E,E,False
3,6,26,6,L,LEU,0,A,E,E,False
4,7,27,7,G,GLY,0,A,E,E,False
...,...,...,...,...,...,...,...,...,...,...
327,330,350,330,P,PRO,0,A,H,H,False
328,331,351,331,R,ARG,0,A,H,H,False
329,332,352,332,L,LEU,0,A,H,H,False
330,333,353,333,R,ARG,0,A,H,H,False


In [50]:
distance_map.residues_i[317:334]

,id,seqres_id,coord_id,one_letter_code,three_letter_code,chain_index,chain_id,sec_struct,sec_struct_3state,hetatm
317,320,340,320,F,PHE,0,A,H,H,False
318,321,341,321,L,LEU,0,A,H,H,False
319,322,342,322,E,GLU,0,A,H,H,False
320,323,343,323,L,LEU,0,A,H,H,False
321,324,344,324,F,PHE,0,A,H,H,False
322,325,345,325,Q,GLN,0,A,H,H,False
323,326,346,326,S,SER,0,A,H,H,False
324,327,347,327,D,ASP,0,A,H,H,False
325,328,348,328,L,LEU,0,A,T,C,False
326,329,349,329,A,ALA,0,A,H,H,False


In [26]:
gene_subset['phenotype'].value_counts()

phenotype
Unknown        105
Susceptible      2
Name: count, dtype: int64

In [30]:
np.unique(gene_subset['position'])

array([  2,   7,  10,  18,  19,  33,  35,  37,  45,  47,  52,  54,  56,
        64,  66,  81,  86,  90,  93, 118, 119, 127, 136, 137, 139, 163,
       166, 168, 169, 170, 171, 183, 187, 188, 190, 199, 212, 216, 224,
       229, 240, 255, 260, 269, 270, 273, 279, 286, 293, 294, 296, 315,
       323])

In [48]:
gene_subset_sorted = gene_subset.sort_values(by='position', ascending=True)


In [49]:
# Filter and print the row where 'position' equals target
gene_subset_sorted[90:107]

,gene,one_letter_mutation,drug,confidence,delta_z,frequency,phenotype,position,Proximity_to_R_Conferring,Nearest_Mutation_Index,aa_index_dist
3833416,fgd1,Q279P,Clofazimine,3) Uncertain significance,0.047684,0.0,Unknown,279,NaN,NaN,37.500558
3833417,fgd1,Q279P,Delamanid,3) Uncertain significance,0.047684,0.0,Unknown,279,NaN,NaN,37.500558
3833486,fgd1,V286A,Clofazimine,3) Uncertain significance,0.062784,0.0,Unknown,286,NaN,NaN,27.720968
3833487,fgd1,V286A,Delamanid,3) Uncertain significance,0.062784,0.0,Unknown,286,NaN,NaN,27.720968
3833393,fgd1,A293V,Clofazimine,3) Uncertain significance,0.035431,0.0,Unknown,293,NaN,NaN,27.720968
3833489,fgd1,V294A,Delamanid,3) Uncertain significance,0.033061,0.0,Unknown,294,NaN,NaN,27.720968
3833488,fgd1,V294A,Clofazimine,3) Uncertain significance,0.033061,0.0,Unknown,294,NaN,NaN,27.720968
3833463,fgd1,K296E,Delamanid,3) Uncertain significance,0.070248,0.0,Unknown,296,NaN,NaN,24.435330
3833462,fgd1,K296E,Clofazimine,3) Uncertain significance,0.070248,0.0,Unknown,296,NaN,NaN,24.435330
3833461,fgd1,K296R,Delamanid,3) Uncertain significance,0.041196,0.0,Unknown,296,NaN,NaN,22.543944


### calculating llr

In [ ]:
catalog_df['Wildtype_AA'] = catalog_df['one_letter_mutation'].str.extract(r'([a-zA-Z])(?=\d)')
catalog_df['Mutated_AA'] = catalog_df['one_letter_mutation'].str.extract(r'(?<=\d)([a-zA-Z])')

In [ ]:
protein_sequences_df=pd.read_csv('/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/catalog/protein_sequences.csv')

In [224]:
# Load the ESM-2 model and tokenizer
model_name = "facebook/esm2_t30_150M_UR50D" ## for rpoB
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmForMaskedLM.from_pretrained(model_name)

# List of amino acids
amino_acids = list("ACDEFGHIKLMNPQRSTVWY")
start_pos = 1
end_pos = None

In [225]:
# Move the model to GPUs and use DataParallel for multi-GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torch.nn.DataParallel(model)
model = model.to(device)

In [226]:
catalog_df['gene'].unique()

array(['rpoB', 'inhA', 'katG', 'embB', 'pncA', 'gyrA', 'rpsL', 'gid',
       'ethA', 'atpE', 'ddn', 'Rv0678', 'pepQ', 'rplC', 'tlyA', 'gyrB',
       'fgd1'], dtype=object)

In [245]:
# Select the rows where 'llr_score' is NaN
nan_llr_rows = catalog_df['llr_score'].isna()

# Extract the 'gene' names from those rows
genes_with_nan_llr = catalog_df.loc[nan_llr_rows, 'gene']

# Print the gene names
print("Genes with NaN in 'llr_score':")
print(genes_with_nan_llr)

unique_genes_with_nan_llr = genes_with_nan_llr.unique()
print("Unique genes with NaN in 'llr_score':")
print(unique_genes_with_nan_llr)


Genes with NaN in 'llr_score':
2103294    gyrA
2103295    gyrA
2103296    gyrA
2103297    gyrA
2103298    gyrA
           ... 
2104087    gyrA
2104090    gyrA
2104093    gyrA
2104094    gyrA
2104097    gyrA
Name: gene, Length: 474, dtype: object
Unique genes with NaN in 'llr_score':
['gyrA']


In [246]:
gene_name='gyrA'

In [247]:
# Loop over unique genes in catalog_df
print(f"Processing gene: {gene_name}")
    
# Select the subset of rows for the current gene
gene_subset = catalog_df[catalog_df['gene'] == gene_name]

Processing gene: gyrA


In [248]:
gene_subset= gene_subset.sort_values(by='position', ascending=True)
gene_subset

,gene,one_letter_mutation,drug,confidence,delta_z,frequency,phenotype,position,Proximity_to_R_Conferring,Nearest_Mutation_Index,aa_index_dist,Wildtype_AA,Mutated_AA,llr_score
2103997,gyrA,T5A,Moxifloxacin,3) Uncertain significance,0.021836,0.214286,Unknown,5,NaN,NaN,26.416943,T,A,-0.568841
2103996,gyrA,T5A,Levofloxacin,3) Uncertain significance,0.021836,0.230769,Unknown,5,NaN,NaN,26.416943,T,A,-0.568841
2103924,gyrA,P7S,Levofloxacin,3) Uncertain significance,0.026082,0.000000,Unknown,7,39.570202,89.0,33.197953,P,S,0.026141
2103925,gyrA,P7S,Moxifloxacin,3) Uncertain significance,0.026082,0.000000,Unknown,7,39.570202,89.0,33.197953,P,S,0.026141
2103923,gyrA,P7L,Moxifloxacin,3) Uncertain significance,0.029659,0.000000,Unknown,7,39.570202,89.0,49.271705,P,L,-0.590122
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2104012,gyrA,T836R,Levofloxacin,3) Uncertain significance,0.036787,0.166667,Unknown,836,NaN,NaN,29.834579,T,R,NaN
2104013,gyrA,T836R,Moxifloxacin,3) Uncertain significance,0.036787,0.000000,Unknown,836,NaN,NaN,29.834579,T,R,NaN
2104014,gyrA,T836K,Levofloxacin,3) Uncertain significance,0.043143,0.000000,Unknown,836,NaN,NaN,29.637804,T,K,NaN
2104015,gyrA,T836K,Moxifloxacin,3) Uncertain significance,0.043143,0.000000,Unknown,836,NaN,NaN,29.637804,T,K,NaN


In [249]:
# Get the corresponding protein sequence for this gene from protein_sequences_df
protein_row = protein_sequences_df[protein_sequences_df['gene'] == gene_name]

In [250]:
if not protein_row.empty:
    protein_sequence = protein_row['protein_sequence'].values[0]
    print(f"Using protein sequence for {gene_name}: {protein_sequence}")

    # Tokenize the protein sequence (assume tokenizer is already initialized)
    input_ids = tokenizer.encode(protein_sequence, return_tensors="pt")
    input_ids = input_ids.to(device)
    sequence_length = input_ids.shape[1] - 2  # Excluding special tokens
    # Example of handling sequence processing or mutation logic
    # Adjust end position if not specified
    if end_pos is None:
        end_pos = sequence_length

Using protein sequence for gyrA: MTDTTLPPDDSLDRIEPVDIEQEMQRSYIDYAMSVIVGRALPEVRDGLKPVHRRVLYAMFDSGFRPDRSHAKSARSVAETMGNYHPHGDASIYDSLVRMAQPWSLRYPLVDGQGNFGSPGNDPPAAMRYTEARLTPLAMEMLREIDEETVDFIPNYDGRVQEPTVLPSRFPNLLANGSGGIAVGMATNIPPHNLRELADAVFWALENHDADEEETLAAVMGRVKGPDFPTAGLIVGSQGTADAYKTGRGSIRMRGVVEVEEDSRGRTSLVITELPYQVNHDNFITSIAEQVRDGKLAGISNIEDQSSDRVGLRIVIEIKRDAVAKVVINNLYKHTQLQTSFGANMLAIVDGVPRTLRLDQLIRYYVDHQLDVIVRRTTYRLRKANERAHILRGLVKALDALDEVIALIRASETVDIARAGLIELLDIDEIQAQAILDMQLRRLAALERQRIIDDLAKIEAEIADLEDILAKPERQRGIVRDELAEIVDRHGDDRRTRIIAADGDVSDEDLIAREDVVVTITETGYAKRTKTDLYRSQKRGGKGVQGAGLKQDDIVAHFFVCSTHDLILFFTTQGRVYRAKAYDLPEASRTARGQHVANLLAFQPEERIAQVIQIRGYTDAPYLVLATRNGLVKKSKLTDFDSNRSGGIVAVNLRDNDELVGAVLCSAGDDLLLVSANGQSIRFSATDEALRPMGRATSGVQGMRFNIDDRLLSLNVVREGTYLLVATSGGYAKRTAIEEYPVQGRGGKGVLTVMYDRRRGRLVGALIVDDDSELYAVTSGGGVIRTAARQVRKAGRQTKGVRLMNLGEGDTLLAIARNAEESGDDNAVDANGADQTGN


In [ ]:
# Loop over each mutation in the gene subset
for index, row in gene_subset.iterrows():
    position = row['position']  # Ensure position is 0-based for input into model
    wt_residue = row['Wildtype_AA']
    mt_residue = row['Mutated_AA']

    # Mask the target position in the input sequence
    masked_input_ids = input_ids.clone()
    masked_input_ids[0, position + 1] = tokenizer.mask_token_id  # +1 for special token offset

    # Get logits for the masked token
    with torch.no_grad():
        logits = model(masked_input_ids).logits

    # Calculate log probabilities for all residues at the masked position
    probabilities = torch.nn.functional.softmax(logits[0, position + 1], dim=0)  # +1 for special token offset
    log_probabilities = torch.log(probabilities)

    # Convert residues to token IDs
    wt_token_id = tokenizer.convert_tokens_to_ids(wt_residue)
    mt_token_id = tokenizer.convert_tokens_to_ids(mt_residue)

    # Get the log probability of the wild-type residue
    log_prob_wt = log_probabilities[wt_token_id].item()

    # Get the log probability of the mutant residue
    log_prob_mt = log_probabilities[mt_token_id].item()

    # Calculate the log-likelihood ratio (LLR)
    llr_score = log_prob_mt - log_prob_wt
    print(f"LLR Score for {gene_name} at position {position}: {llr_score}")

    # Store the LLR score in your DataFrame
    catalog_df.at[index, 'llr_score'] = llr_score
# else:
#     print(f"No protein sequence found for gene {gene_name}")




In [252]:
# Display the DataFrame with LLR scores
print(catalog_df[['gene', 'one_letter_mutation', 'llr_score']])

         gene one_letter_mutation  llr_score
0        rpoB              A1002P  -0.920767
1        rpoB              A1055P  -0.554745
2        rpoB              A1072G  -0.093131
3        rpoB                A10T  -0.404166
4        rpoB              A1153V  -1.110641
...       ...                 ...        ...
3833489  fgd1               V294A   3.764369
3833490  fgd1                V37G   0.933872
3833491  fgd1                V37G   0.933872
3833492  fgd1               M139I  -1.272278
3833493  fgd1                S56C  -0.623899

[6172 rows x 3 columns]


In [253]:
catalog_df.to_csv("/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/all_proteins_freq_details_proximity_aaindex_llr.csv",index=False)

## feature 6: thermostability score

In [254]:
catalog_df=pd.read_csv("/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/all_proteins_freq_details_proximity_aaindex_llr.csv")

In [284]:
gene_name = 'pncA'

In [285]:
# Load the Rosetta score CSV
rosetta_scores_path = f'/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/Rosetta/{gene_name}_ddG.csv'  # Update with the correct path
rosetta_scores = pd.read_csv(rosetta_scores_path)

In [286]:
# Select the subset of rows for the current gene
gene_subset = catalog_df[catalog_df['gene'] == gene_name]
gene_subset= gene_subset.sort_values(by='position', ascending=True)
gene_subset

,gene,one_letter_mutation,drug,confidence,delta_z,frequency,phenotype,position,Proximity_to_R_Conferring,Nearest_Mutation_Index,aa_index_dist,Wildtype_AA,Mutated_AA,llr_score
2603,pncA,R2P,Pyrazinamide,3) Uncertain significance,0.142795,1.000000,Unknown,2,1.349375,3.0,42.601487,M,E,1.194902
2602,pncA,R2L,Pyrazinamide,3) Uncertain significance,0.189905,0.000000,Unknown,2,1.349375,3.0,40.578571,M,D,1.905348
2601,pncA,R2Q,Pyrazinamide,3) Uncertain significance,0.118815,0.000000,Unknown,2,1.349375,3.0,22.596164,M,S,1.070911
2604,pncA,R2W,Pyrazinamide,3) Uncertain significance,0.223304,0.500000,Unknown,2,1.349375,3.0,38.456975,M,C,1.564430
2585,pncA,A3E,Pyrazinamide,2) Assoc w R - Interim,0.178938,1.000000,Resistant,3,1.348999,4.0,28.301032,F,E,-5.166997
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2901,pncA,V180A,Pyrazinamide,1) Assoc w R,0.084864,1.000000,Resistant,180,NaN,NaN,27.720968,E,L,1.417449
2750,pncA,L182W,Pyrazinamide,1) Assoc w R,0.072667,1.000000,Resistant,182,NaN,NaN,33.439645,T,L,-3.299496
2748,pncA,L182F,Pyrazinamide,3) Uncertain significance,0.057356,0.200000,Unknown,182,NaN,NaN,22.837724,T,R,-2.306675
2749,pncA,L182S,Pyrazinamide,1) Assoc w R,0.113166,0.789474,Resistant,182,NaN,NaN,37.326490,I,M,-0.805004


In [287]:
rosetta_scores['Wildtype_AA'] = rosetta_scores['variant'].str.extract(r'([a-zA-Z])(?=\d)')
rosetta_scores['position'] = rosetta_scores['variant'].str.extract(r'(\d+)').astype(int)
rosetta_scores['Mutated_AA'] = rosetta_scores['variant'].str.extract(r'(?<=\d)([a-zA-Z])')

In [288]:
# Create a function to adjust the number based on conditions
def adjust_number(index):
    if index < 30:  # Assuming 23 + 7
        return index- 7
    elif 30 <= index <= 1180:  # Assuming 1173 + 7
        return index - 6
    else:
        return index


In [289]:
# Apply the adjustment to the 'Number' in pdb_df
rosetta_scores['adjusted_position'] = rosetta_scores['position'].apply(adjust_number)

In [290]:
# Merge the dataframes on Adjusted_Number and One_Letter
common_data = pd.merge(
    rosetta_scores,
    gene_subset,
    left_on=['Wildtype_AA', 'adjusted_position','Mutated_AA'],
    right_on=['Wildtype_AA', 'position','Mutated_AA'],
    how='inner'
)

In [291]:
common_data

,fa_atr,fa_rep,fa_sol,fa_intra_rep,fa_intra_sol_xover4,lk_ball_wtd,fa_elec,pro_close,hbond_sr_bb,hbond_lr_bb,...,drug,confidence,delta_z,frequency,phenotype,position_y,Proximity_to_R_Conferring,Nearest_Mutation_Index,aa_index_dist,llr_score
0,3.703,-0.366,-6.006,-0.003,-1.187,-0.528,6.636,0.000,0.000,0.0,...,Pyrazinamide,2) Assoc w R - Interim,0.359740,1.0,Resistant,8,1.341352,9.0,29.775078,1.866321
1,-9.806,337.971,-6.145,0.019,0.153,-0.248,4.788,0.000,0.000,0.0,...,Pyrazinamide,3) Uncertain significance,0.092732,0.0,Unknown,41,4.493794,90.0,31.887645,0.619807
2,-1.210,-0.320,2.143,0.018,1.229,0.464,-0.009,-1.815,0.000,0.0,...,Pyrazinamide,3) Uncertain significance,0.085859,0.0,Unknown,56,1.319556,57.0,29.775078,-0.793363
3,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,...,Pyrazinamide,1) Assoc w R,0.148057,1.0,Resistant,138,1.320682,139.0,40.517919,-7.158254
4,-13.347,748.403,9.020,0.035,0.867,0.029,0.505,0.000,0.000,0.0,...,Pyrazinamide,3) Uncertain significance,0.147404,1.0,Unknown,156,1.328520,155.0,49.271705,0.115352
5,-4.833,330.334,-0.478,0.000,0.129,0.850,0.837,21.834,0.995,0.0,...,Pyrazinamide,2) Assoc w R - Interim,0.079402,1.0,Resistant,162,1.326742,163.0,34.619038,-1.368432
6,-2.395,0.199,0.760,0.005,0.359,-0.013,1.260,0.000,0.000,0.0,...,Pyrazinamide,3) Uncertain significance,0.071116,0.0,Unknown,162,1.326742,163.0,26.911929,0.617904
7,-2.089,14.588,-3.419,0.011,-0.009,0.349,1.865,0.000,0.000,0.0,...,Pyrazinamide,3) Uncertain significance,0.085823,1.0,Unknown,162,1.326742,163.0,44.149248,-0.601446


In [292]:
common_data=common_data.drop(columns=['fa_atr', 'fa_rep', 'fa_sol', 'fa_intra_rep', 'fa_intra_sol_xover4',
       'lk_ball_wtd', 'fa_elec', 'pro_close', 'hbond_sr_bb', 'hbond_lr_bb',
       'hbond_bb_sc', 'hbond_sc', 'dslf_fa13', 'omega', 'fa_dun', 'p_aa_pp',
       'yhh_planarity', 'ref', 'rama_prepro','variant','position_x'])

In [293]:
common_data=common_data.rename(columns={"score": "thermostability"})

In [294]:
# Move 'gene' column to the front
cols = ['gene'] + [col for col in common_data.columns if col != 'gene']
common_data = common_data[cols]


In [296]:
common_data

,gene,thermostability,Wildtype_AA,Mutated_AA,adjusted_position,one_letter_mutation,drug,confidence,delta_z,frequency,phenotype,position_y,Proximity_to_R_Conferring,Nearest_Mutation_Index,aa_index_dist,llr_score
0,pncA,2.790980,E,K,8,D8H,Pyrazinamide,2) Assoc w R - Interim,0.359740,1.0,Resistant,8,1.341352,9.0,29.775078,1.866321
1,pncA,330.814295,T,F,41,Y41C,Pyrazinamide,3) Uncertain significance,0.092732,0.0,Unknown,41,4.493794,90.0,31.887645,0.619807
2,pncA,6.366237,P,Q,56,D56H,Pyrazinamide,3) Uncertain significance,0.085859,0.0,Unknown,56,1.319556,57.0,29.775078,-0.793363
3,pncA,0.000000,E,E,138,C138R,Pyrazinamide,1) Assoc w R,0.148057,1.0,Resistant,138,1.320682,139.0,40.517919,-7.158254
4,pncA,748.997716,G,R,156,L156P,Pyrazinamide,3) Uncertain significance,0.147404,1.0,Unknown,156,1.328520,155.0,49.271705,0.115352
5,pncA,363.352241,T,P,162,G162D,Pyrazinamide,2) Assoc w R - Interim,0.079402,1.0,Resistant,162,1.326742,163.0,34.619038,-1.368432
6,pncA,0.782315,T,R,162,G162S,Pyrazinamide,3) Uncertain significance,0.071116,0.0,Unknown,162,1.326742,163.0,26.911929,0.617904
7,pncA,12.635194,T,V,162,G162R,Pyrazinamide,3) Uncertain significance,0.085823,1.0,Unknown,162,1.326742,163.0,44.149248,-0.601446


In [297]:
common_data.to_csv(f'/work/pi_annagreen_umass_edu/mahbuba/resistance_forecast/data/derived_features/{gene_name}_thermostability.csv', index=False)